# EHR-FM Tutorial: Pipeline Walkthrough

This notebook walks through the EHR-FM pipeline on a small synthetic dataset:

1. **Inspect** the pre-generated synthetic MEDS data
2. **Train vocabularies** in both joint and factorized modes
3. **Pretokenize** the data using each vocabulary
4. **Train** a tiny transformer model
5. **Compute patient representations** from the trained model

All outputs are written to `tutorials/output/` (gitignored). The synthetic data (generated using lives in
`tutorials/synthetic_data/` and is committed to the repo.

**Prerequisites:** `pip install -e .` from the ehr-fm repo root.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import polars as pl

TUTORIALS_DIR = Path(os.path.abspath("")).resolve()
if TUTORIALS_DIR.name != "tutorials":
    TUTORIALS_DIR = TUTORIALS_DIR / "tutorials"
os.chdir(TUTORIALS_DIR)

SYNTHETIC_DATA = TUTORIALS_DIR / "synthetic_data"
MEDS_DIR = SYNTHETIC_DATA / "meds"
MEDS_READER_DIR = SYNTHETIC_DATA / "meds_reader"
OUTPUT_DIR = TUTORIALS_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)


def run(cmd: list[str], description: str = "") -> None:
    """Run a shell command, print its stdout, and raise on failure."""
    if description:
        print(f"\n{'=' * 60}")
        print(f"  {description}")
        print(f"{'=' * 60}")
    print(f"$ {' '.join(str(c) for c in cmd)}\n")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-2000:])
    if result.returncode != 0:
        print(f"STDERR:\n{result.stderr[-3000:]}")
        raise RuntimeError(f"Command failed (rc={result.returncode})")


print(f"Working directory : {TUTORIALS_DIR}")
print(f"Synthetic data    : {SYNTHETIC_DATA}")
print(f"Output directory  : {OUTPUT_DIR}")

---
## 0. Inspect Synthetic Data

The synthetic dataset was generated by `generate_synthetic_data.py` and contains
~200 patients with birth events, sex demographics, lab results (with numeric values),
diagnoses, and drug orders (with text values describing route of administration).

The data exists in two formats:
- **MEDS** (`synthetic_data/meds/`) -- standard parquet with `data/` and `metadata/` directories
- **MEDS Reader** (`synthetic_data/meds_reader/`) -- optimized columnar format used by EHR-FM

In [ ]:
events = pl.read_parquet(MEDS_DIR / "data" / "0.parquet")
samples = pl.read_parquet(MEDS_DIR / "metadata" / "samples.parquet")

print(f"Patients : {events['subject_id'].n_unique()}")
print(f"Events   : {len(events)}")
print(f"Columns  : {events.columns}")
print()

print("--- Event counts by code prefix ---")
code_counts = (
    events.with_columns(pl.col("code").str.split("/").list.first().alias("prefix"))
    .group_by("prefix")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
print(code_counts)
print()

print("--- Sample rows ---")
print(events.head(10))
print()

print("--- Train / validation split ---")
print(samples.group_by("split").agg(pl.len().alias("count")))

---
## 1. Train Vocabulary -- Joint Mode

In **joint** tokenization, each unique (code, value-bucket) combination becomes a single
token.  For example, a glucose lab with a numeric value in the 3rd quantile becomes
`LAB/glucose/Q:3`.  This produces compact sequences (one token per event) but a larger
vocabulary.

We train the vocabulary on the MEDS Reader data with a small `--vocab_size 8192` cap
(sufficient for this tiny dataset).

In [ ]:
VOCAB_JOINT_DIR = OUTPUT_DIR / "vocab_joint"

run(
    [
        "train_vocabulary",
        "--dataset_path", str(MEDS_READER_DIR),
        "--out_dir", str(VOCAB_JOINT_DIR),
        "--tokenization_mode", "joint",
        "--vocab_size", "8192",
    ],
    description="Training joint vocabulary",
)

In [ ]:
from ehr_fm.io import read_json_yaml

vocab_joint = read_json_yaml(VOCAB_JOINT_DIR / "vocab.json")

entries = vocab_joint["vocab"]
print(f"Joint vocabulary size: {len(entries)} tokens")
print(f"Age stats: mean={vocab_joint['age_stats']['mean']:.1f}, std={vocab_joint['age_stats']['std']:.1f}")
print()

print("--- First 20 vocabulary entries ---")
for i, entry in enumerate(entries[:20]):
    print(f"  [{i:3d}] {entry}")

print(f"\n  ... ({len(entries) - 20} more entries)")

---
## 2. Train Vocabulary -- Factorized Mode

In **factorized** tokenization, codes and their attributes are separate tokens.
A glucose lab becomes *two* tokens: `glucose` (the base code) and `Q:3` (the quantile
bucket).  Drugs get a base code plus a `TXT:oral` token.  This keeps the vocabulary
smaller (shared quantile / text / stage tokens) at the cost of longer sequences.

In [ ]:
VOCAB_FACT_DIR = OUTPUT_DIR / "vocab_factorized"

run(
    [
        "train_vocabulary",
        "--dataset_path", str(MEDS_READER_DIR),
        "--out_dir", str(VOCAB_FACT_DIR),
        "--tokenization_mode", "factorized",
        "--vocab_size", "8192",
    ],
    description="Training factorized vocabulary",
)

In [ ]:
vocab_fact = read_json_yaml(VOCAB_FACT_DIR / "vocab.json")

entries_f = vocab_fact["vocab"]
print(f"Factorized vocabulary size: {len(entries_f)} tokens")
print(f"Joint vocabulary size:      {len(entries)} tokens")
print()

types_f = {}
for e in entries_f:
    t = e.get("type", "unknown")
    types_f[t] = types_f.get(t, 0) + 1
print("--- Factorized token types ---")
for t, c in sorted(types_f.items(), key=lambda x: -x[1]):
    print(f"  {t:20s} : {c}")

print("\n--- First 30 factorized entries ---")
for i, entry in enumerate(entries_f[:30]):
    print(f"  [{i:3d}] {entry}")

---
## 3. Pretokenize

Pretokenization converts raw MEDS events into integer token sequences using the trained
vocabulary.  We tokenize both **train** and **validation** splits for each vocabulary mode.

The output is a parquet file per split with columns:
- `subject_id` -- patient identifier
- `index_time` -- prediction time
- `token_ids` -- list of integer token IDs
- `age` -- list of ages (in days) per token
- `age_normalized` -- normalized ages
- `length` -- sequence length

In [ ]:
TOKENS_JOINT_DIR = OUTPUT_DIR / "tokens_joint"

for split in ["train", "validation"]:
    out = TOKENS_JOINT_DIR / split
    run(
        [
            "pretokenize",
            "--dataset_path", str(MEDS_READER_DIR),
            "--vocab_path", str(VOCAB_JOINT_DIR / "vocab.json"),
            "--out_dir", str(out),
            "--split", split,
            "--workers", "0",
        ],
        description=f"Pretokenize joint -- {split}",
    )

In [ ]:
TOKENS_FACT_DIR = OUTPUT_DIR / "tokens_factorized"

for split in ["train", "validation"]:
    out = TOKENS_FACT_DIR / split
    run(
        [
            "pretokenize",
            "--dataset_path", str(MEDS_READER_DIR),
            "--vocab_path", str(VOCAB_FACT_DIR / "vocab.json"),
            "--out_dir", str(out),
            "--split", split,
            "--workers", "0",
        ],
        description=f"Pretokenize factorized -- {split}",
    )

In [ ]:
tok_joint = pl.read_parquet(TOKENS_JOINT_DIR / "train" / "patients_tokenized.parquet")
tok_fact = pl.read_parquet(TOKENS_FACT_DIR / "train" / "patients_tokenized.parquet")

print(f"Joint tokenized  -- patients: {len(tok_joint)}, columns: {tok_joint.columns}")
print(f"Factorized tokenized -- patients: {len(tok_fact)}, columns: {tok_fact.columns}")
print()

print("--- Joint: sequence lengths ---")
print(tok_joint.select("subject_id", "length").describe())
print()

print("--- Factorized: sequence lengths ---")
print(tok_fact.select("subject_id", "length").describe())
print()

# Decode one patient's tokens back to vocab entries
row = tok_joint.row(0, named=True)
token_ids = row["token_ids"]
print(f"--- Patient {row['subject_id']}: first 15 tokens (joint) ---")
for tid in token_ids[:15]:
    entry = entries[tid] if tid < len(entries) else {"unknown": tid}
    print(f"  token_id={tid:4d}  ->  {entry}")

---
## 4. Train Model

We train a very small EHR-FM transformer on the joint-tokenized training
data.  The architecture is intentionally tiny so it runs quickly on CPU:

| Parameter | Value | Production typical |
|-----------|-------|-------------------|
| hidden_size | 64 | 768 |
| n_layers | 2 | 22 -- 28 |
| n_heads | 4 | 12 |
| max_steps | 50 | 3-5 epochs |

The model is trained with next-token prediction (NTP) as the self-supervised objective.

In [ ]:
actual_vocab_size = len(entries)

# Pad vocab_size to next multiple of 64
padded_vocab_size = ((actual_vocab_size + 63) // 64) * 64

config = {
    "vocab_size": padded_vocab_size,
    "hidden_size": 64,
    "intermediate_size": 96,
    "n_layers": 2,
    "n_heads": 4,
    "attention_width": 64,
    "num_ntp_classes": actual_vocab_size,
    "max_steps": 50,
    "max_tokens_per_batch": 2048,
    "min_patients_per_batch": 2,
    "learning_rate": 1e-3,
    "logging_steps": 10,
    "eval_strategy": "steps",
    "evaluation_strategy": "steps",
    "eval_steps": 25,
    "save_steps": 25,
    "save_total_limit": 2,
    "report_to": "none",
    "optimizer_name": "adamw",
    "load_best_model_at_end": False,
}

config_path = OUTPUT_DIR / "train_config.yaml"
import yaml

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Config written to {config_path}")
print(f"  vocab_size (padded) : {padded_vocab_size}")
print(f"  num_ntp_classes     : {actual_vocab_size}")
print()

MODEL_DIR = OUTPUT_DIR / "model"

run(
    [
        "train_ehr_fm", str(config_path),
        "--train_path", str(TOKENS_JOINT_DIR / "train" / "patients_tokenized.parquet"),
        "--val_path", str(TOKENS_JOINT_DIR / "validation" / "patients_tokenized.parquet"),
        "--output_dir", str(MODEL_DIR),
    ],
    description="Training EHR-FM model (tiny, ~50 steps)",
)

In [ ]:
print("--- Model output directory ---")
for item in sorted(MODEL_DIR.iterdir()):
    if item.is_dir():
        contents = list(item.iterdir())
        print(f"  {item.name}/  ({len(contents)} files)")
    else:
        print(f"  {item.name}  ({item.stat().st_size:,} bytes)")

---
## 5. Compute Representations

Given a trained model and pretokenized data, `compute_representations` extracts the
last hidden state for each patient as a dense embedding vector.  These representations
can then be used for downstream tasks (linear probing, few-shot classification, etc.).

In [ ]:
# Find the latest checkpoint
checkpoints = sorted(
    [d for d in MODEL_DIR.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
    key=lambda p: int(p.name.split("-")[1]),
)
checkpoint_path = checkpoints[-1]
print(f"Using checkpoint: {checkpoint_path.name}")

REPR_PATH = OUTPUT_DIR / "representations.parquet"

run(
    [
        "compute_representations",
        "--pretokenized_dataset_path", str(TOKENS_JOINT_DIR / "validation" / "patients_tokenized.parquet"),
        "--model_path", str(checkpoint_path),
        "--output_path", str(REPR_PATH),
        "--max_tokens_per_batch", "4096",
    ],
    description="Computing patient representations",
)

In [ ]:
import numpy as np

repr_df = pl.read_parquet(REPR_PATH)
print(f"Representations shape: {len(repr_df)} patients x {len(repr_df['representations'][0])}d")
print()
print(repr_df.head(5).select("id", "index_t"))

---
## 6. Summary

This tutorial walked through the full EHR-FM pipeline:

```
MEDS data  -->  train_vocabulary  -->  pretokenize  -->  train_ehr_fm  -->  compute_representations
```

**What we covered:**

| Step | CLI command | Output |
|------|-------------|--------|
| Vocabulary (joint) | `train_vocabulary --tokenization_mode joint` | `vocab.json` |
| Vocabulary (factorized) | `train_vocabulary --tokenization_mode factorized` | `vocab.json` |
| Pretokenize | `pretokenize` | `patients_tokenized.parquet` |
| Train | `train_ehr_fm config.yaml` | model checkpoints |
| Representations | `compute_representations` | `representations.parquet` |

**For real-world usage**, scale up:

- **Larger model:** `hidden_size=768`, `n_layers=22-28`, `n_heads=12`
- **More training:** `3--5 epochs`, learning rate scheduling
- **GPU:** Use `--bf16` or `--fp16`, increase `max_tokens_per_batch`
- **Parallel tokenization:** `--workers -1` uses all CPU cores

See the [CLI documentation](../ehr_fm/scripts/README.md) for detailed argument reference.